In [1]:
#@title CELL 1 — Cài thư viện, Java 11 và kiểm tra bộ chấm điểm
"""Level 4 • đánh giá lại checkpoint có sẵn seeds 42/44/46.
Chọn GPU, chạy Cell 1, rồi chọn seed ở Cell 2 và chạy các cell còn lại.
Beam k=3, alpha=.6, max30 gồmBOS; cùng thuật toán normalized_candidates_v1 của Lv1.
Giữ mô hình và biến thể retrieval w0p6_0p4 đã chọn trên validation.
Không train lại, không dọn/move/xóa/checkpoint cũ. Kết quả mới riêng từng seed.
Seed đo độ biến thiên fine-tuning Lv4 từ cùng init Lv3 seed42, không phải paired end-to-end.
Mã đã kiểm tra CPU; chưa chạy toàn bộ 5000 ảnh/GPU trong phiên tạo notebook.
"""
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import sys, subprocess, shutil, tempfile, tarfile, urllib.request, zipfile
from pathlib import Path
from importlib import metadata

RUNTIME_DIR = Path("/content/level1_beam3_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pycocoevalcap==1.2", "nltk>=3.9,<4", "tqdm>=4.66,<5"],
    check=True,
)

def java11_version(java):
    result = subprocess.run([str(java), "-version"], capture_output=True, text=True)
    return result.returncode == 0 and 'version "11.' in result.stderr

java_candidates = list(Path("/usr/lib/jvm").glob("java-11-*/bin/java"))
java_candidates += list((RUNTIME_DIR / "java11").glob("*/bin/java"))
if not any(java11_version(p) for p in java_candidates):
    try:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq",
                        "openjdk-11-jre-headless"], check=True)
    except subprocess.CalledProcessError:
        # Colab Ubuntu mới có thể không cung cấp package OpenJDK 11.
        java_root = RUNTIME_DIR / "java11"
        java_root.mkdir(exist_ok=True)
        with tempfile.TemporaryDirectory(dir=RUNTIME_DIR) as tmp:
            archive = Path(tmp) / "jdk11.tar.gz"
            urllib.request.urlretrieve(
                "https://api.adoptium.net/v3/binary/latest/11/ga/linux/x64/jre/hotspot/normal/eclipse",
                archive,
            )
            with tarfile.open(archive, "r:gz") as tar:
                tar.extractall(java_root, filter="data")
    java_candidates = list(Path("/usr/lib/jvm").glob("java-11-*/bin/java"))
    java_candidates += list((RUNTIME_DIR / "java11").glob("*/bin/java"))
JAVA_BIN = next((p for p in java_candidates if java11_version(p)), None)
if JAVA_BIN is None:
    raise RuntimeError("Không cài được Java 11; xem log cài đặt ở trên.")
os.environ["JAVA_HOME"] = str(JAVA_BIN.parent.parent)
os.environ["PATH"] = str(JAVA_BIN.parent) + os.pathsep + os.environ["PATH"]
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

import nltk
for resource in ("punkt", "punkt_tab", "wordnet", "omw-1.4"):
    if not nltk.download(resource, quiet=True):
        raise RuntimeError(f"Không tải được NLTK resource: {resource}")

import pycocoevalcap.spice.spice as spice_module
SPICE_DIR = Path(spice_module.__file__).parent
SPICE_LIB = SPICE_DIR / "lib"
SPICE_LIB.mkdir(exist_ok=True)
# Tải cả code jar và models jar qua HTTPS, tránh downloader HTTP cũ.
for suffix in ("", "-models"):
    jar_name = f"stanford-corenlp-3.6.0{suffix}.jar"
    jar_path = SPICE_LIB / jar_name
    if not jar_path.exists() or not zipfile.is_zipfile(jar_path):
        url = ("https://repo.maven.apache.org/maven2/edu/stanford/nlp/"
               f"stanford-corenlp/3.6.0/{jar_name}")
        with tempfile.TemporaryDirectory(dir=RUNTIME_DIR) as tmp:
            downloaded = Path(tmp) / jar_name
            urllib.request.urlretrieve(url, downloaded)
            if not zipfile.is_zipfile(downloaded):
                raise RuntimeError(f"Jar tải về không hợp lệ: {jar_name}")
            shutil.copyfile(downloaded, jar_path)

# Package và cache SPICE đều nằm trong runtime; không ghi công cụ lên Drive.
spice_module.TEMP_DIR = str(RUNTIME_DIR / "spice_tmp")
spice_module.CACHE_DIR = str(RUNTIME_DIR / "spice_cache")
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.spice.spice import Spice
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

class ManagedMeteor(Meteor):
    def close(self):
        process = getattr(self, "meteor_p", None)
        if process is not None and process.poll() is None:
            try:
                if process.stdin is not None:
                    process.stdin.close()
            except (OSError, ValueError):
                pass
            finally:
                process.kill()
                process.wait(timeout=30)
    def __del__(self):
        self.close()

def close_meteor(scorer):
    scorer.close()

_smoke_gts = PTBTokenizer().tokenize({1: [{"caption": "a cat sits on a mat"}]})
_smoke_res = PTBTokenizer().tokenize({1: [{"caption": "a cat sits on a mat"}]})
_meteor = ManagedMeteor()
try:
    _meteor_score, _ = _meteor.compute_score(_smoke_gts, _smoke_res)
finally:
    close_meteor(_meteor)
del _meteor
_spice_score, _ = Spice().compute_score(_smoke_gts, _smoke_res)
import math
if not all(math.isfinite(float(x)) for x in (_meteor_score, _spice_score)):
    raise RuntimeError("Smoke test METEOR/SPICE thất bại.")
print(f"Bộ chấm điểm OK • METEOR={_meteor_score:.4f} • SPICE={_spice_score:.4f}")
print("pycocoevalcap", metadata.version("pycocoevalcap"), "| NLTK", nltk.__version__)
# Nguồn triển khai: https://github.com/salaniz/pycocoevalcap


openjdk version "11.0.32.1" 2026-08-18
OpenJDK Runtime Environment (build 11.0.32.1+1-post-1ubuntu1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 11.0.32.1+1-post-1ubuntu1-24.04-Ubuntu, mixed mode, sharing)

Bộ chấm điểm OK • METEOR=1.0000 • SPICE=1.0000
pycocoevalcap 1.2 | NLTK 3.9.1


In [2]:
#@title CELL 2 — Chọn seed và đọc dữ liệu/checkpoint Level 4 đã có
DATA_PREFLIGHT_OK = False
SEED_TO_RUN = "42" #@param ["42", "44", "46"]
EXPERIMENT_TAG = "main" #@param {type:"string"}
OLD_SESSION_STOPPED = False #@param {type:"boolean"}
SELECTED_SEED = int(SEED_TO_RUN)
from google.colab import drive
drive.mount("/content/drive")
import json, re, hashlib, gc, time, platform, random, math
from collections import defaultdict
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import display, Markdown

if not re.fullmatch(r"[A-Za-z0-9_-]{1,40}", EXPERIMENT_TAG):
    raise ValueError("Tag chỉ gồm chữ không dấu, số, _ hoặc -, dài 1–40 ký tự.")
DATA_ROOT = Path("/content/drive/MyDrive/DataKhoaLuan/Data")
BASE_DIR = DATA_ROOT / "baseline_level4"
OUTPUT_ROOT = BASE_DIR / "multiseed_beam3_v1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Bật GPU Colab trước khi đánh giá 5.000 ảnh.")
MAX_LEN = MAX_LENGTH = 30
D_MODEL, NUM_LAYERS, NUM_HEADS, DIM_FEEDFORWARD, DROPOUT, TAU = 256, 3, 8, 1024, 0.1, 1.0
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3
SOURCE_CHECKPOINTS = {s: BASE_DIR / "full" / f"Lan01_seed{s}" / "best_model.pt"
                      for s in (42, 44, 46)}
SOURCE_SHA256 = {
    42: "5f37df6cf1134b243e9bcfca4a9f73493c51c047b68363c8b8fda3006445d583",
    44: "3e4751568c83c7178896e9223db85164e39b2e56d0a013a1ee7457e4f8fbe7c8",
    46: "592123e91505f156a4e63de70e9a805e02316c8aaa6073bef36762ffa7f0be17",
}
SEED42_CHECKPOINT = SOURCE_CHECKPOINTS[42]

def read_json(path):
    with Path(path).open(encoding="utf-8") as f:
        return json.load(f)
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()
def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
        separators=(",", ":")).encode("utf-8")).hexdigest()
DATA_FINGERPRINTS = {}
def fingerprint(path, label, expected=None):
    if not Path(path).is_file():
        raise FileNotFoundError(f"Thiếu input đã có; không tự tạo/sửa: {path}")
    digest = sha256_file(path)
    if expected is not None and digest != expected:
        raise ValueError(f"Hash input khác bản nhóm đã xác minh: {path}")
    DATA_FINGERPRINTS[label] = digest
    return digest

vocab_path = DATA_ROOT / "baseline_level1/vocab.json"
fingerprint(vocab_path, "vocab",
    "c47a5de5309a21a22a5e9267505c8397781584c49eb29a626a3f8c595b6bdf83")
vocab = read_json(vocab_path)
stoi = {str(k): int(v) for k, v in vocab["stoi"].items()}
itos = (list(vocab["itos"]) if isinstance(vocab["itos"], list) else
        [vocab["itos"][str(i)] for i in range(len(stoi))])
VOCAB_SIZE = len(stoi)
assert VOCAB_SIZE == 16214 and all(itos[i] == w for w, i in stoi.items())
assert all(stoi.get(t) == i for i, t in enumerate(("<PAD>", "<BOS>", "<EOS>", "<UNK>")))

ann_dirs = [DATA_ROOT / "COCO_2014/annotations_trainval2014/annotations",
            DATA_ROOT / "COCO_2014/annotations"]
ANNOTATION_DIR = next((p for p in ann_dirs if
    (p / "captions_karpathy_test.json").is_file()), None)
if ANNOTATION_DIR is None:
    raise FileNotFoundError("Thiếu annotations Karpathy.")
splits, captions = {}, {}
for split in ("train", "val", "test"):
    p = ANNOTATION_DIR / f"captions_karpathy_{split}.json"
    fingerprint(p, f"annotations/{split}")
    payload = read_json(p)
    idx = defaultdict(list)
    for a in payload["annotations"]:
        if str(a["caption"]).strip():
            idx[int(a["image_id"])].append(str(a["caption"]))
    splits[split] = set(idx)
    captions[split] = dict(idx) if split == "test" else None
    assert len(idx) == (113287 if split == "train" else 5000)
    assert sum(map(len, idx.values())) == (566747 if split == "train" else 25010)
    assert set(idx) == {int(im["id"]) for im in payload["images"]}
    del payload, idx
assert not (splits["train"] & splits["val"] or splits["train"] & splits["test"]
            or splits["val"] & splits["test"])
references = captions["test"]

feature_path = DATA_ROOT / "baseline_level1/clip_image_features/test_features.npy"
ids_path = feature_path.with_name("test_image_ids.json")
fingerprint(feature_path, "visual/test")
fingerprint(ids_path, "image_ids/test")
visual = np.load(feature_path, allow_pickle=False)
ids = [int(x) for x in read_json(ids_path)]
assert visual.shape == (5000, 512) and visual.dtype == np.float32
assert np.isfinite(visual).all() and len(set(ids)) == 5000 and set(ids) == splits["test"]
assert np.allclose(np.linalg.norm(visual, axis=1), 1, atol=1e-3)
image_ids = {"test": ids}

KB_DIR = DATA_ROOT / "knowledge_base_level1_karpathy"
kb_path = KB_DIR / "text_embeddings_train_restval.npy"
mapping_path = KB_DIR / "text_embedding_mapping.jsonl"
print("Kiểm tra hash KB (file lớn, chỉ đọc)...")
fingerprint(kb_path, "kb/embeddings",
    "a7d701abb4efbd23ea548b8499785a4fc19a26070a0686d271e0ace17f9db225")
fingerprint(mapping_path, "kb/mapping",
    "91f8a7dd2b70f88ec14c0ab6cc67ec9de2dd2b32062d903d0698801f53b7cf32")
kb = np.load(kb_path, mmap_mode="r", allow_pickle=False)
assert kb.shape == (566747, 512) and kb.dtype == np.float32
kb_image_ids = []
with mapping_path.open(encoding="utf-8") as f:
    for position, line in enumerate(f):
        row = json.loads(line)
        assert int(row.get("row_id", position)) == position
        assert row.get("split") in ("train", "restval")
        kb_image_ids.append(int(row["image_id"]))
assert len(kb_image_ids) == len(kb)
assert set(kb_image_ids) == splits["train"]
assert not (set(kb_image_ids) & (splits["val"] | splits["test"]))

# Hai bản NPZ này có cùng toàn bộ model inputs; chỉ scalar weight_config_hash đổi.
# Đã chứng minh bằng cách đổi riêng scalar trong RAM, giữ ZIP64 headers và khôi phục
# chính xác SHA256 của cả file lịch sử. Không chấp nhận hash khác hoặc ghi lại nguồn.
VERIFIED_TEST_CACHE_VERSIONS = {
    "5df0ac35cf48fa81f1ff67dc97149d7aa4d64a2b9cfeda2a7674f0f56114c097":
        "8b90e22fe5ef4d42f3c1c395e0876aef2e1bb3beaadb7a7ef09b8bb44c295b3b",
    "6619333ebc6e73d5956f393644634d025f9be705a6e9e466175b671d36f6f935":
        "1115e13973de6a5579006c66f1fae625baa2de6cdb22b26549dddee8f371ec30",
}
RETRIEVAL_CACHE_PROVENANCE = {
    "compatibility": "metadata-only weight_config_hash refresh; all other NPZ member bytes identical",
    "proof": "replacing only that scalar in memory with original ZIP64 headers reproduces full historical file SHA256",
    "verified_versions": VERIFIED_TEST_CACHE_VERSIONS,
}
def verified_test_cache_hash(path):
    actual = sha256_file(path)
    expected_weight_hash = VERIFIED_TEST_CACHE_VERSIONS.get(actual)
    if expected_weight_hash is None:
        raise ValueError(f"Cache khác các bản đã chứng minh tương đương; giữ nguyên và dừng: {path}")
    with np.load(path, allow_pickle=False) as candidate:
        if ("weight_config_hash" not in candidate
                or candidate["weight_config_hash"].shape != ()
                or str(candidate["weight_config_hash"].item()) != expected_weight_hash):
            raise ValueError(f"Metadata cache không khớp phiên bản SHA đã khóa: {path}")
    if sha256_file(path) != actual:
        raise RuntimeError(f"Cache thay đổi trong lúc kiểm tra; giữ nguyên và dừng: {path}")
    return actual

cache_candidates = [
    DATA_ROOT / "baseline_level3/nguoi_A_level3/level3/full_top50_k5/final_cache/w0p6_0p4",
    DATA_ROOT / "baseline_level3/nguoi_B_level3/inputs/reranked_cache/w0p6_0p4",
]
cache_dirs = [p for p in cache_candidates if
              (p / "reranked_top5_test_w0p6_0p4.npz").is_file()]
if not cache_dirs:
    raise FileNotFoundError("Thiếu cache w0p6_0p4 đã có; không tự đổi biến thể hay tạo lại.")
cache_path = cache_dirs[0] / "reranked_top5_test_w0p6_0p4.npz"
# Kiểm tra MỌI bản trùng đường dẫn ứng viên. Chỉ hai SHA đã chứng minh ở trên
# mới được cùng tồn tại; một bản khác bất kỳ vẫn làm dừng trước khi tạo đầu ra.
cache_hashes = [verified_test_cache_hash(p / cache_path.name) for p in cache_dirs]
selected_cache_sha256 = cache_hashes[0]
fingerprint(cache_path, "retrieval/test", selected_cache_sha256)
with np.load(cache_path, allow_pickle=False) as cache:
    cached_ids = [int(x) for x in cache["image_ids"]]
    rows = np.asarray(cache["row_indices"], dtype=np.int64)
    scores = np.asarray(cache["final_scores"] if "final_scores" in cache else cache["scores"],
                        dtype=np.float32)
if sha256_file(cache_path) != selected_cache_sha256:
    raise RuntimeError("Cache thay đổi trong lúc nạp; giữ nguyên và dừng.")
assert len(cached_ids) == len(set(cached_ids)) == 5000 and set(cached_ids) == set(ids)
assert rows.shape == scores.shape == (5000, 5)
assert (rows >= 0).all() and (rows < len(kb)).all()
assert (np.diff(np.sort(rows, axis=1), axis=1) != 0).all()
assert np.isfinite(scores).all() and (scores >= -1e-6).all() and (scores <= 1 + 1e-6).all()
cache_by_id = {iid: i for i, iid in enumerate(cached_ids)}
order = np.asarray([cache_by_id[i] for i in ids])
rows, scores = rows[order], scores[order]
knowledge = np.asarray(kb[rows], dtype=np.float32)
assert knowledge.shape == (5000, 5, 512) and np.isfinite(knowledge).all()
assert not (set(np.asarray(kb_image_ids)[rows].ravel()) & splits["test"])
packed = np.zeros((5000, 7, 512), dtype=np.float32)
packed[:, 0] = visual
packed[:, 1:6] = knowledge
packed[:, 6, :5] = scores
features = {"test": packed}

class ImageDataset(Dataset):
    def __len__(self):
        return len(image_ids["test"])
    def __getitem__(self, index):
        return image_ids["test"][index], torch.from_numpy(features["test"][index])
test_loader = DataLoader(ImageDataset(), batch_size=1, shuffle=False,
                        drop_last=False, num_workers=0, pin_memory=True)
source_path = SOURCE_CHECKPOINTS[SELECTED_SEED]
if not source_path.is_file() or sha256_file(source_path) != SOURCE_SHA256[SELECTED_SEED]:
    raise ValueError(f"Checkpoint seed {SELECTED_SEED} thiếu/khác hash đã xác minh: {source_path}")
print(f"Đã đọc test=5000 ảnh/25010 captions; seed {SELECTED_SEED} có checkpoint gốc.")
print("Notebook chỉ đánh giá lại; không train và không sửa full/Lan01_seed*/.")

HISTORICAL_METRICS = {}
for seed in (42, 44, 46):
    metrics_path = SOURCE_CHECKPOINTS[seed].parent / "test_metrics_R3_beam.json"
    history_path = SOURCE_CHECKPOINTS[seed].parent / "training_history.json"
    fingerprint(metrics_path, f"source_metrics/{seed}")
    fingerprint(history_path, f"source_history/{seed}")
    previous = read_json(metrics_path)
    if not isinstance(previous.get("CIDEr"), (int, float)) or not math.isfinite(previous["CIDEr"]):
        raise ValueError(f"Thiếu bằng chứng test hoàn tất của seed {seed}.")
    history = read_json(history_path)
    if not isinstance(history, dict) or not history.get("train_loss") or not history.get("val_cider"):
        raise ValueError(f"Thiếu lịch sử train của seed {seed}.")
    if len(history["train_loss"]) != len(history["val_cider"]):
        raise ValueError(f"History seed {seed} không hợp lệ.")
    HISTORICAL_METRICS[seed] = previous
    print(f"Seed {seed} • CIDEr cũ ×100: {previous['CIDEr'] * 100:.4f}")
DATA_PREFLIGHT_OK = True


Mounted at /content/drive
Kiểm tra hash KB (file lớn, chỉ đọc)...
Đã đọc test=5000 ảnh/25010 captions; seed 42 có checkpoint gốc.
Notebook chỉ đánh giá lại; không train và không sửa full/Lan01_seed*/.
Seed 42 • CIDEr cũ ×100: 109.8646
Seed 44 • CIDEr cũ ×100: 109.6173
Seed 46 • CIDEr cũ ×100: 111.1489


In [3]:
#@title CELL 3 — Model nguyên bản, Beam Search chung và vùng ghi an toàn
if not globals().get("DATA_PREFLIGHT_OK", False):
    raise RuntimeError("Cell 2 chưa hoàn tất kiểm tra dữ liệu. Chạy lại Cell 2 thành công rồi mới chạy Cell 3; không bỏ qua lỗi cache.")
# ==============================================================================
# CELL 9 — KIẾN TRÚC MÔ HÌNH LEVEL 4: GATED FUSION & MULTI-TOKEN KNOWLEDGE
# ==============================================================================

class GatedFusion(nn.Module):
    """1. CƠ CHẾ GATED FUSION:
    Sinh hệ số cổng vô hướng g ∈ [0, 1] từ ảnh và trung bình tri thức.
    Đầu vào: [v_norm (512); k_agg_norm (512); confidence (1)] = 1025 chiều.
    Tầng Linear: Linear(1025, 1) với đúng 1,026 tham số.
    Zero-Init (W=0, b=0) đảm bảo g ban đầu = 0.5000 chính xác!"""
    def __init__(self, clip_dim=512):
        super().__init__()
        self.gate_linear = nn.Linear(clip_dim * 2 + 1, 1)
        nn.init.zeros_(self.gate_linear.weight)
        nn.init.zeros_(self.gate_linear.bias)
        self.sigmoid = nn.Sigmoid()

    def forward(self, v_img, k_tokens, scores):
        v_norm = F.normalize(v_img, p=2, dim=-1)
        # Softmax pooling trên 5 câu truy hồi
        w = F.softmax(scores / TAU, dim=-1).unsqueeze(1) # [B, 1, 5]
        k_agg = torch.bmm(w, k_tokens).squeeze(1)        # [B, 512]
        k_agg_norm = F.normalize(k_agg, p=2, dim=-1)
        conf = scores.mean(dim=-1, keepdim=True)         # [B, 1]

        gate_input = torch.cat([v_norm, k_agg_norm, conf], dim=-1) # [B, 1025]
        g = self.sigmoid(self.gate_linear(gate_input))             # [B, 1]
        return g

class DualBranchDecoderLayer(nn.Module):
    """2. CƠ CHẾ MULTI-TOKEN DUAL-BRANCH DECODER:
    - Nhánh Visual Cross-Attention: Chú ý vào 1 token ảnh (m_vis).
    - Nhánh Knowledge Cross-Attention: Chú ý vào 5 token tri thức riêng rẽ (m_knw).
    - Dung hợp tại tầng ngữ cảnh: c_fused = (1 - g) * c_vis + g * c_knw."""
    def __init__(self, d_model=256, nhead=8, dim_feedforward=1024, dropout=0.1):
        super().__init__()
        self.self_attn  = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.cross_vis  = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.cross_knw  = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)

        self.linear1    = nn.Linear(d_model, dim_feedforward)
        self.linear2    = nn.Linear(dim_feedforward, d_model)
        self.activation = nn.ReLU()

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, tgt, m_vis, m_knw, gate, tgt_mask=None, tgt_pad_mask=None):
        # 1. Self-Attention
        h_self = self.norm1(tgt + self.drop(self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask, key_padding_mask=tgt_pad_mask)[0]))
        # 2. Dual Cross-Attentions
        c_vis = self.cross_vis(h_self, m_vis, m_vis)[0] # Ngữ cảnh thị giác
        c_knw = self.cross_knw(h_self, m_knw, m_knw)[0] # Ngữ cảnh tri thức đa token
        # 3. Gated Blending
        g_expand = gate.unsqueeze(1) # [B, 1, 1]
        c_fused  = (1.0 - g_expand) * c_vis + g_expand * c_knw
        h_cross  = self.norm2(h_self + self.drop(c_fused))
        # 4. Feed-Forward Network
        out = self.norm3(h_cross + self.drop(self.linear2(self.drop(self.activation(self.linear1(h_cross))))))
        return out

class Level4_RAGCaptioner(nn.Module):
    """MÔ HÌNH CHÍNH LEVEL 4 HOÀN CHỈNH."""
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_layers=NUM_LAYERS, nhead=NUM_HEADS, d_ff=DIM_FEEDFORWARD, dropout=DROPOUT, max_len=MAX_LENGTH):
        super().__init__()
        self.token_embedding   = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(max_len, d_model)

        self.visual_proj       = nn.Linear(512, d_model)
        self.knowledge_proj    = nn.Linear(512, d_model)
        self.gated_fusion      = GatedFusion(clip_dim=512)

        self.layers = nn.ModuleList([DualBranchDecoderLayer(d_model, nhead, d_ff, dropout) for _ in range(num_layers)])
        self.output_projection = nn.Linear(d_model, vocab_size)

    def forward(self, v_img, k_tokens, scores, input_ids):
        B, T = input_ids.shape
        g = self.gated_fusion(v_img, k_tokens, scores) # [B, 1]

        m_vis = self.visual_proj(v_img).unsqueeze(1)   # [B, 1, 256]
        m_knw = self.knowledge_proj(k_tokens)          # [B, 5, 256] (Multi-token Memory)

        pos = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, -1)
        h = self.token_embedding(input_ids) + self.position_embedding(pos)
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=input_ids.device), diagonal=1)
        pad_mask = input_ids.eq(PAD_ID)

        for layer in self.layers:
            h = layer(h, m_vis, m_knw, g, tgt_mask=mask, tgt_pad_mask=pad_mask)

        logits = self.output_projection(h)
        return logits, g


class Level4PackedCaptioner(Level4_RAGCaptioner):
    def forward(self, packed_inputs, input_ids):
        logits, gate = super().forward(packed_inputs[:,0], packed_inputs[:,1:6],
                                      packed_inputs[:,6,:5], input_ids)
        return logits

def new_model():
    return Level4PackedCaptioner().to(DEVICE)

@torch.inference_mode()
def beam_search_decode(model, image_feature, beam_size=3, max_len=30, alpha=0.6):
    """max_len gồm BOS; mỗi bước xếp hạng cả beam EOS và beam đang mở rộng."""
    if beam_size < 1 or not 2 <= max_len <= model.position_embedding.num_embeddings:
        raise ValueError("beam_size >= 1 và 2 <= max_len <= 30.")
    model.eval()
    device = next(model.parameters()).device
    image_feature = torch.as_tensor(image_feature, dtype=torch.float32, device=device)
    if image_feature.shape != (7, 512):
        raise ValueError(f"Cần packed input (7,512), nhận {image_feature.shape}.")
    beams = [([BOS_ID], 0.0)]
    def normalized(beam):
        tokens, score = beam
        return score / (((5.0 + len(tokens) - 1) / 6.0) ** alpha)
    for _ in range(max_len - 1):
        active = [(tokens, score) for tokens, score in beams if tokens[-1] != EOS_ID]
        if not active:
            break
        candidates = [(tokens, score) for tokens, score in beams if tokens[-1] == EOS_ID]
        inputs = torch.tensor([tokens for tokens, _ in active], device=device)
        memory = image_feature.unsqueeze(0).expand(len(active), -1, -1)
        logits = model(memory, inputs)[:, -1, :].float()
        if not torch.isfinite(logits).all():
            raise RuntimeError("Model sinh logits không hữu hạn; dừng để kiểm tra checkpoint.")
        logits[:, [PAD_ID, BOS_ID]] = -torch.inf
        log_prob = torch.log_softmax(logits, dim=-1)
        top_scores, top_ids = log_prob.topk(min(beam_size, VOCAB_SIZE - 2), dim=-1)
        # Một lần chuyển GPU -> CPU cho toàn bộ beam ở bước này.
        for (tokens, score), values, indices in zip(
                active, top_scores.tolist(), top_ids.tolist()):
            candidates.extend((tokens + [int(token)], score + float(value))
                              for value, token in zip(values, indices))
        beams = sorted(candidates, key=normalized, reverse=True)[:beam_size]
        if all(tokens[-1] == EOS_ID for tokens, _ in beams):
            break
    best_tokens, _ = max(beams, key=normalized)
    return " ".join(itos[t] for t in best_tokens if t not in (PAD_ID, BOS_ID, EOS_ID))

ARCHITECTURE = {
    "name": "Level4_GatedFusion_DualBranch_native5",
    "clip_dim": 512, "d_model": 256, "num_layers": 3, "nhead": 8,
    "dim_feedforward": 1024, "dropout": 0.1, "max_length": MAX_LEN,
    "vocab_size": VOCAB_SIZE, "gate": "Linear(1025,1)", "knowledge_tokens": 5,
}
DECODE_CONFIG = {"method": "beam_search", "beam_size": 3, "alpha": 0.6,
    "max_len": MAX_LEN, "max_len_includes_bos": True, "mask_pad_bos": True,
    "implementation": "normalized_candidates_v1", "precision": "float32", "tf32": False}
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
PROTOCOL = {
    "level": 4, "action": "reevaluate_existing_3_seeds", "experiment_tag": EXPERIMENT_TAG,
    "architecture": ARCHITECTURE, "decode": DECODE_CONFIG, "data": DATA_FINGERPRINTS,
    "source_checkpoints_sha256": {str(k): v for k, v in SOURCE_SHA256.items()},
    "retrieval": {"variant": "w0p6_0p4", "top_k": 5, "temperature": 1.0,
                  "selected_on": "historical_validation_only",
                  "cache_provenance": RETRIEVAL_CACHE_PROVENANCE},
    "evaluator": "pycocoevalcap==1.2/PTB/Java11",
    "storage_schema": "per_seed_guarded_v2",
    "seed_scope": "Lv4 fine-tuning seeds from common historical Lv3 seed42; not paired end-to-end",
    "old_training_recipe": {"max_epochs": 15, "batch_size": 64, "learning_rate": 5e-5,
        "weight_decay": 1e-4, "scheduler": "cosine_eta_min_1e-6", "patience": 4,
        "selection": "greedy_validation_CIDEr"},
}
PROTOCOL_HASH = canonical_hash(PROTOCOL)
METRIC_NAMES = ["Bleu_1","Bleu_2","Bleu_3","Bleu_4","METEOR","ROUGE_L","CIDEr","SPICE"]
# Mọi đường dẫn ghi đều bị giới hạn vào RUN_DIR; input gốc chỉ được đọc.
import uuid
from contextlib import contextmanager
RUN_DIR = OUTPUT_ROOT / ("run_" + PROTOCOL_HASH[:16])
RESULT_DIR = RUN_DIR
_OBSERVED_FILES = {}
_ACTIVE_LOCKS = {}

def output_path(path):
    path = Path(path).resolve()
    try:
        path.relative_to(RUN_DIR.resolve())
    except ValueError as exc:
        raise PermissionError(f"Từ chối ghi ngoài vùng kết quả mới: {path}") from exc
    return path

def owned_meta(seed, role):
    return {"protocol_hash": PROTOCOL_HASH, "storage_schema": 2,
            "seed": seed, "role": role}

def verify_owner(payload, seed, role):
    expected = owned_meta(seed, role)
    meta = payload.get("_meta", {}) if isinstance(payload, dict) else {}
    if any(meta.get(k) != v for k, v in expected.items()):
        raise ValueError(f"File khác seed/protocol/định dạng ({role}); giữ nguyên và dừng.")

def remember_file(path, before):
    path = output_path(path)
    after = sha256_file(path)
    if before != after:
        raise RuntimeError(f"File đang bị phiên khác thay đổi: {path.name}")
    _OBSERVED_FILES[str(path)] = after

def read_owned_json(path, seed, role):
    path = output_path(path)
    if not path.exists():
        _OBSERVED_FILES[str(path)] = None
        return None
    if not path.is_file():
        raise FileExistsError(f"Đường dẫn đã là thư mục: {path}")
    before = sha256_file(path)
    payload = read_json(path)  # JSON lỗi: dừng, không tự ghi đè.
    verify_owner(payload, seed, role)
    remember_file(path, before)
    return payload

def seed_dir(seed, create=False):
    if seed not in (42, 44, 46):
        raise ValueError("Chỉ có seed 42, 44, 46.")
    path = output_path(RUN_DIR / f"seed_{seed}")
    if create:
        path.mkdir(parents=True, exist_ok=True)
    return path

def check_lock(scope):
    lock = _ACTIVE_LOCKS.get(scope)
    if lock is None:
        raise RuntimeError(f"Thiếu khóa ghi cho {scope}.")
    path, token = lock
    try:
        current = read_json(path)
    except (OSError, ValueError) as exc:
        raise RuntimeError("Khóa bị mất hoặc hỏng; dừng ghi.") from exc
    if current.get("token") != token:
        raise RuntimeError("Quyền ghi đã thay đổi; dừng để bảo vệ dữ liệu.")

@contextmanager
def writing_scope(scope):
    global OLD_SESSION_STOPPED
    # Một phiên đang chạy cho mỗi seed. Khóa trên Drive là lớp phòng vệ,
    # không thay thế quy tắc không chạy hai Colab VM cùng seed.
    folder = RUN_DIR if scope == "summary" else seed_dir(scope, create=True)
    lock_path = output_path(folder / ".active_run.json")
    if scope in _ACTIVE_LOCKS:
        raise RuntimeError(f"Seed/phần tổng hợp {scope} đang chạy trong phiên này.")
    if lock_path.exists():
        old_hash = sha256_file(lock_path)
        try:
            old = read_json(lock_path)
        except (OSError, ValueError):
            old = None
        if not OLD_SESSION_STOPPED:
            raise RuntimeError(
                f"Còn phiên/khóa cũ của {scope}. Nếu phiên cũ đã DỪNG HẲN, bật "
                "OLD_SESSION_STOPPED ở Cell 2, rồi chạy lại. Không chạy hai phiên cùng seed.")
        if old is not None:
            verify_owner(old, scope, "active_run")
        elif lock_path.stat().st_size > 4096:
            raise RuntimeError("File khóa hỏng có kích thước bất thường; giữ nguyên để kiểm tra.")
        if sha256_file(lock_path) != old_hash:
            raise RuntimeError("Khóa vừa đổi; không thu hồi.")
        lock_path.unlink()
        OLD_SESSION_STOPPED = False  # Thu hồi một lần, tránh vô tình lấy khóa khác.
    token = uuid.uuid4().hex
    record = {"_meta": owned_meta(scope, "active_run"), "token": token,
              "started_unix": time.time()}
    with lock_path.open("x", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False)
    _ACTIVE_LOCKS[scope] = (lock_path, token)
    try:
        check_lock(scope)
        yield
    finally:
        try:
            if lock_path.is_file() and read_json(lock_path).get("token") == token:
                lock_path.unlink()
        finally:
            _ACTIVE_LOCKS.pop(scope, None)

def required_scope(path):
    relative = output_path(path).relative_to(RUN_DIR.resolve())
    if len(relative.parts) > 1 and relative.parts[0].startswith("seed_"):
        return int(relative.parts[0].split("_")[1])
    return "summary"

def commit_file(path, write_temp, mutable=False):
    path = output_path(path)
    scope = required_scope(path)
    check_lock(scope)
    key = str(path)
    if key not in _OBSERVED_FILES:
        if path.exists():
            raise FileExistsError(f"Chưa xác minh file có sẵn; KHÔNG ghi đè: {path}")
        _OBSERVED_FILES[key] = None
    expected = _OBSERVED_FILES[key]
    current = sha256_file(path) if path.exists() else None
    if current != expected:
        raise RuntimeError(f"File đã đổi sau khi đọc; KHÔNG ghi đè: {path}")
    tmp = output_path(path.with_name("." + path.name + "." + uuid.uuid4().hex + ".tmp"))
    try:
        write_temp(tmp)
        new_hash = sha256_file(tmp)
        if current == new_hash:
            return False
        if current is not None and not mutable:
            raise FileExistsError(f"Kết quả đã có và khác nội dung; giữ nguyên: {path}")
        check_lock(scope)
        if (sha256_file(path) if path.exists() else None) != expected:
            raise RuntimeError(f"File vừa đổi; KHÔNG ghi đè: {path}")
        # Chỉ thay trạng thái đang chạy của đúng experiment/seed đã xác minh.
        # Không di chuyển hay đổi tên bất kỳ dữ liệu/checkpoint nguồn nào.
        os.replace(tmp, path)
        _OBSERVED_FILES[key] = new_hash
        if sha256_file(path) != new_hash:
            raise IOError(f"Readback sau lưu không khớp: {path}")
        return True
    finally:
        # Chỉ xóa file tạm UUID của chính lần ghi này.
        tmp.unlink(missing_ok=True)

def save_json(value, path, mutable=False):
    def write_temp(tmp):
        with tmp.open("x", encoding="utf-8") as f:
            json.dump(value, f, ensure_ascii=False, sort_keys=True,
                      indent=2, allow_nan=False)
            f.flush()
            os.fsync(f.fileno())
    return commit_file(path, write_temp, mutable=mutable)

def initialize_experiment():
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    manifest_path = RUN_DIR / "experiment.json"
    manifest = {"_meta": owned_meta(None, "experiment"),
                "protocol": PROTOCOL, "seeds": [42, 44, 46]}
    def validate_manifest():
        last_error = None
        for attempt in range(6):
            try:
                previous = read_owned_json(manifest_path, None, "experiment")
                if previous is not None:
                    if previous != manifest:
                        raise ValueError("Manifest khác cấu hình; không dùng chung thư mục.")
                    return
            except (OSError, json.JSONDecodeError) as exc:
                last_error = exc
            if attempt < 5:
                time.sleep(0.2)
        raise RuntimeError(
            "Manifest thiếu/hỏng hoặc chưa ghi xong; mọi file được giữ nguyên. "
            "Thử chạy lại Cell 3. Nếu vẫn lỗi, đổi EXPERIMENT_TAG ở Cell 2 để "
            "tạo thí nghiệm mới, không xóa/ghi đè thư mục cũ.") from last_error

    if manifest_path.exists():
        validate_manifest()
    elif any(RUN_DIR.iterdir()):
        # Seed khác có thể vừa khởi tạo manifest giữa exists() và iterdir().
        validate_manifest()
    else:
        try:
            with manifest_path.open("x", encoding="utf-8") as f:
                json.dump(manifest, f, ensure_ascii=False, sort_keys=True, indent=2)
                f.flush()
                os.fsync(f.fileno())
        except FileExistsError:
            validate_manifest()
    print("Kết quả mới:", RUN_DIR)
    print(f"Seed đang chọn: {SELECTED_SEED} →", seed_dir(SELECTED_SEED))
    print("Checkpoint/vocab/features/annotations gốc: CHỈ ĐỌC.")

def load_checkpoint(path, expected_seed=None):
    path = Path(path)
    seed = 42 if expected_seed is None else int(expected_seed)
    if path.resolve() != SOURCE_CHECKPOINTS[seed].resolve():
        raise PermissionError("Notebook này chỉ đọc checkpoint gốc đã xác minh của seed đang chọn.")
    before = sha256_file(path)
    if before != SOURCE_SHA256[seed]:
        raise ValueError(f"Checkpoint seed {seed} khác hash; giữ nguyên, không tự thay file.")
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    except Exception as exc:
        raise RuntimeError(f"Không nạp được checkpoint đã xác minh: {exc}") from exc
    if not isinstance(checkpoint, dict) or "model_state_dict" not in checkpoint:
        raise ValueError("Checkpoint không chứa model_state_dict.")
    expected_epoch_zero_based = {42: 6, 44: 2, 46: 3}
    if int(checkpoint.get("epoch", -1)) != expected_epoch_zero_based[seed]:
        raise ValueError("Best epoch không khớp bản nhóm đã kiểm tra.")
    if not math.isfinite(float(checkpoint.get("val_cider", float("nan")))):
        raise ValueError("Checkpoint thiếu validation CIDEr hữu hạn.")
    if sha256_file(path) != before:
        raise RuntimeError("Checkpoint bị đổi trong lúc đọc.")
    # Export best lịch sử không có cờ completion/seed. Hash đã đối chiếu
    # cùng đúng thư mục Lan01_seedN và bằng chứng test/history gốc.
    checkpoint["training_complete"] = True
    checkpoint["_historical_source_verified"] = True
    return checkpoint

def model_from_checkpoint(path, expected_seed=None):
    checkpoint = load_checkpoint(path, expected_seed)
    model = new_model()
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    return model.eval()

_model_check = new_model().eval()
assert sum(p.numel() for p in _model_check.parameters()) == 12538968
with torch.inference_mode():
    assert _model_check(torch.zeros(1,7,512,device=DEVICE), torch.tensor([[BOS_ID]],device=DEVICE)).shape == (1,1,VOCAB_SIZE)
del _model_check
torch.cuda.empty_cache()
initialize_experiment()


Kết quả mới: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level4/multiseed_beam3_v1/run_0ba507b683ac43ef
Seed đang chọn: 42 → /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level4/multiseed_beam3_v1/run_0ba507b683ac43ef/seed_42
Checkpoint/vocab/features/annotations gốc: CHỈ ĐỌC.


In [4]:
#@title CELL 4 — Hàm đánh giá chung, tự tiếp tục predictions và giữ kết quả cuối
def prediction_prefix_valid(predictions, complete=False):
    if not isinstance(predictions, list) or len(predictions) > len(image_ids["test"]):
        return False
    try:
        ids = [int(row["image_id"]) for row in predictions]
        valid = (ids == image_ids["test"][:len(ids)]
                 and len(ids) == len(set(ids))
                 and all(isinstance(row["caption"], str) for row in predictions))
        return valid and (not complete or len(ids) == 5000)
    except (KeyError, TypeError, ValueError):
        return False

def valid_metrics(metrics):
    return all(name in metrics and isinstance(metrics[name], (int, float))
               and math.isfinite(float(metrics[name])) for name in METRIC_NAMES)

_TOKENIZED_REFERENCES = None
def score_predictions(predictions):
    global _TOKENIZED_REFERENCES
    if not prediction_prefix_valid(predictions, complete=True):
        raise ValueError("Predictions phải phủ đúng 5.000 test IDs theo thứ tự cache.")
    if _TOKENIZED_REFERENCES is None:
        print("PTB tokenize toàn bộ 25.010 reference captions...")
        _TOKENIZED_REFERENCES = PTBTokenizer().tokenize({
            i: [{"caption": caption} for caption in references[i]]
            for i in image_ids["test"]})
    prediction_by_id = {int(p["image_id"]): p["caption"] for p in predictions}
    res = PTBTokenizer().tokenize({
        i: [{"caption": prediction_by_id[i]}] for i in image_ids["test"]})
    metrics = {}
    scorers = [(lambda: Bleu(4), METRIC_NAMES[:4]),
               (ManagedMeteor, ["METEOR"]), (Rouge, ["ROUGE_L"]),
               (Cider, ["CIDEr"]), (Spice, ["SPICE"])]
    for factory, names in scorers:
        print("Đang chấm:", ", ".join(names), flush=True)
        scorer = factory()
        try:
            score, _ = scorer.compute_score(_TOKENIZED_REFERENCES, res)
            for name, value in zip(names, score if len(names) > 1 else [score]):
                value = float(value)
                if not math.isfinite(value):
                    raise ValueError(f"{name} không hữu hạn.")
                metrics[name] = value
        except Exception as exc:
            raise RuntimeError(f"Chấm {names} thất bại; giữ predictions để chấm lại: {exc}") from exc
        finally:
            if isinstance(scorer, ManagedMeteor):
                scorer.close()
            del scorer
            gc.collect()
    if not valid_metrics(metrics):
        raise RuntimeError("Chưa đủ 8 chỉ số hợp lệ; không ghi điểm giả.")
    return metrics

def load_prediction_packet(path, seed, signature):
    packet = read_owned_json(path, seed, "predictions")
    if packet is None:
        return None
    meta = packet["_meta"]
    predictions = packet.get("predictions")
    if meta.get("signature") != signature:
        raise ValueError("Predictions khác checkpoint/cấu hình; GIỮ NGUYÊN, không ghi đè.")
    complete = meta.get("complete")
    if not isinstance(complete, bool) or not prediction_prefix_valid(predictions, complete):
        raise ValueError("Predictions hỏng/trùng/sai thứ tự ID; giữ nguyên và dừng.")
    if meta.get("predictions_hash") != canonical_hash(predictions):
        raise ValueError("Nội dung predictions đã đổi; giữ nguyên và dừng.")
    return packet

def evaluate_seed(seed, checkpoint_path):
    check_lock(seed)
    checkpoint_path = Path(checkpoint_path)
    checkpoint_hash = sha256_file(checkpoint_path)
    checked = load_checkpoint(checkpoint_path, None if seed == 42 else seed)
    if seed != 42 and not checked.get("training_complete", False):
        raise RuntimeError("Checkpoint chưa hoàn tất train; không đánh giá như kết quả cuối.")
    del checked
    metrics_path = seed_dir(seed) / f"metrics_seed{seed}.json"
    predictions_path = seed_dir(seed) / f"predictions_seed{seed}.json"
    signature = canonical_hash({"seed": seed, "checkpoint_sha256": checkpoint_hash,
                               "protocol_hash": PROTOCOL_HASH})
    packet = load_prediction_packet(predictions_path, seed, signature)
    previous = read_owned_json(metrics_path, seed, "metrics")
    if previous is not None:
        if (previous["_meta"].get("signature") != signature or not valid_metrics(previous)
                or previous["_meta"].get("status") != "complete"
                or packet is None or not packet["_meta"]["complete"]
                or previous["_meta"].get("predictions_sha256") != sha256_file(predictions_path)):
            raise ValueError("Có metrics nhưng không xác minh được; GIỮ NGUYÊN và dừng.")
        if sha256_file(checkpoint_path) != checkpoint_hash:
            raise RuntimeError("Checkpoint vừa đổi trong lúc kiểm tra; giữ nguyên kết quả.")
        print(f"Seed {seed}: đã đủ kết quả hợp lệ; bỏ qua, không sửa file.")
        return previous

    predictions = [] if packet is None else list(packet["predictions"])
    start = time.perf_counter()
    def persist_predictions(complete):
        nonlocal packet
        if packet is not None and packet["_meta"]["complete"]:
            raise FileExistsError("Predictions đã hoàn tất; không thay thế.")
        if not prediction_prefix_valid(predictions, complete):
            raise ValueError("Không ghi predictions sai coverage.")
        updated = {
            "_meta": {**owned_meta(seed, "predictions"), "signature": signature,
                "checkpoint_sha256": checkpoint_hash, "complete": complete,
                "num_images": len(predictions),
                "predictions_hash": canonical_hash(predictions),
                "format": "envelope; COCO rows are in the predictions key"},
            "predictions": predictions,
        }
        save_json(updated, predictions_path, mutable=packet is not None)
        # Copy để trạng thái đã lưu không đổi theo list đang append.
        packet = {"_meta": dict(updated["_meta"]), "predictions": list(predictions)}

    if packet is None or not packet["_meta"]["complete"]:
        print(f"Seed {seed}: tiếp tục Beam Search từ ảnh {len(predictions) + 1}/5000.")
        model = model_from_checkpoint(checkpoint_path, None if seed == 42 else seed)
        if sha256_file(checkpoint_path) != checkpoint_hash:
            del model
            raise RuntimeError("Checkpoint đổi trong lúc nạp; dừng.")
        last_saved_count = len(predictions)
        try:
            with tqdm(total=5000, initial=len(predictions),
                      desc=f"Seed {seed} • Beam k=3", unit="ảnh") as progress:
                for index, (ids, batch_features) in enumerate(test_loader):
                    if index < len(predictions):
                        continue
                    assert int(ids[0]) == image_ids["test"][index]
                    caption = beam_search_decode(model, batch_features[0],
                        beam_size=3, max_len=MAX_LEN, alpha=0.6)
                    predictions.append({"image_id": int(ids[0]), "caption": caption})
                    progress.update(1)
                    if len(predictions) % 100 == 0 and len(predictions) < 5000:
                        persist_predictions(complete=False)
                        last_saved_count = len(predictions)
            if sha256_file(checkpoint_path) != checkpoint_hash:
                raise RuntimeError("Checkpoint đã đổi trong khi decode; không chấm điểm lẫn phiên.")
            persist_predictions(complete=True)
        except BaseException:
            if len(predictions) > last_saved_count and len(predictions) < 5000:
                try:
                    persist_predictions(complete=False)
                except Exception as save_error:
                    print("Không lưu thêm được tiến độ; bản đã lưu trước vẫn giữ:", save_error)
            raise
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()
    else:
        print(f"Seed {seed}: đã có 5.000 predictions; chỉ chạy phần chấm điểm.")

    if sha256_file(checkpoint_path) != checkpoint_hash:
        raise RuntimeError("Checkpoint đã đổi; không chấm/lưu kết quả cuối.")
    prediction_hash = sha256_file(predictions_path)
    metrics = score_predictions(predictions)
    if (sha256_file(checkpoint_path) != checkpoint_hash
            or sha256_file(predictions_path) != prediction_hash):
        raise RuntimeError("Input đánh giá đã đổi trong lúc chấm; không ghi metrics.")
    metrics["_meta"] = {
        **owned_meta(seed, "metrics"), "status": "complete", "signature": signature,
        "split": "MS COCO 2014 Karpathy Test", "num_images": 5000,
        "num_references": sum(map(len, references.values())),
        "checkpoint": str(checkpoint_path), "checkpoint_sha256": checkpoint_hash,
        "predictions_sha256": prediction_hash, "decoding": DECODE_CONFIG,
        "metric_scale": "raw pycocoevalcap scores, no multiplication by 100",
        "elapsed_seconds_this_session": time.perf_counter() - start,
        "environment": {"python": platform.python_version(), "torch": str(torch.__version__),
            "numpy": np.__version__, "pycocoevalcap": metadata.version("pycocoevalcap"),
            "gpu": torch.cuda.get_device_name(), "java": "11"},
    }
    save_json(metrics, metrics_path)  # Immutable final: không cho thay file khác.
    print(f"Seed {seed} hoàn tất:", {k: round(metrics[k], 6) for k in METRIC_NAMES})
    return metrics



In [5]:
#@title CELL 5 — Đánh giá lại seed Level 4 đang chọn (không train)
with writing_scope(SELECTED_SEED):
    selected_metrics = evaluate_seed(SELECTED_SEED, SOURCE_CHECKPOINTS[SELECTED_SEED])
print("CIDEr cũ ×100:", HISTORICAL_METRICS[SELECTED_SEED]["CIDEr"] * 100)
print("CIDEr decode chung ×100:", selected_metrics["CIDEr"] * 100)
print("Nếu điểm thay đổi, đó là đổi decoder/evaluator trên cùng weights; không phải train mới.")


Seed 42: tiếp tục Beam Search từ ảnh 1/5000.


Seed 42 • Beam k=3:   0%|          | 0/5000 [00:00<?, ?ảnh/s]

PTB tokenize toàn bộ 25.010 reference captions...
Đang chấm: Bleu_1, Bleu_2, Bleu_3, Bleu_4
{'testlen': 48231, 'reflen': 47772, 'guess': [48231, 43231, 38231, 33231], 'correct': [35780, 19557, 10073, 5201]}
ratio: 1.0096081386586073
Đang chấm: METEOR
Đang chấm: ROUGE_L
Đang chấm: CIDEr
Đang chấm: SPICE
Seed 42 hoàn tất: {'Bleu_1': 0.741847, 'Bleu_2': 0.579309, 'Bleu_3': 0.445507, 'Bleu_4': 0.342986, 'METEOR': 0.275125, 'ROUGE_L': 0.557653, 'CIDEr': 1.108336, 'SPICE': 0.202995}
CIDEr cũ ×100: 109.86458809559285
CIDEr decode chung ×100: 110.83358422124165
Nếu điểm thay đổi, đó là đổi decoder/evaluator trên cùng weights; không phải train mới.


In [6]:
#@title CELL 6 — Trạng thái từng seed; chỉ tổng hợp khi đủ 3 kết quả cuối
def summarize_completed_seeds():
    ALL_SEEDS = [42, 44, 46]
    seed_metrics = {}
    source_hashes = {}
    for seed in ALL_SEEDS:
        path = seed_dir(seed) / f"metrics_seed{seed}.json"
        prediction_path = seed_dir(seed) / f"predictions_seed{seed}.json"
        if not path.is_file() or not prediction_path.is_file():
            raise FileNotFoundError(f"Thiếu kết quả seed {seed}; hãy chạy Cell 4/5 trước.")
        row = read_owned_json(path, seed, "metrics")
        meta = row.get("_meta", {})
        if not valid_metrics(row) or meta.get("seed") != seed:
            raise ValueError(f"Điểm seed {seed} thiếu hoặc không hợp lệ.")
        if meta.get("protocol_hash") != PROTOCOL_HASH:
            raise ValueError(f"Seed {seed} dùng protocol khác, không được gộp.")
        expected_checkpoint = SOURCE_CHECKPOINTS[seed]
        if Path(meta.get("checkpoint", "")).resolve() != expected_checkpoint.resolve():
            raise ValueError(f"Seed {seed} trỏ nhầm checkpoint.")
        signature = canonical_hash({"seed": seed,
            "checkpoint_sha256": meta["checkpoint_sha256"], "protocol_hash": PROTOCOL_HASH})
        if meta.get("signature") != signature or meta.get("status") != "complete":
            raise ValueError(f"Seed {seed} chưa có metrics cuối đúng signature.")
        packet = load_prediction_packet(prediction_path, seed, signature)
        if packet is None or not packet["_meta"]["complete"]:
            raise ValueError(f"Seed {seed} chưa có đủ 5.000 predictions cuối.")
        if meta.get("predictions_sha256") != sha256_file(prediction_path):
            raise ValueError(f"Predictions seed {seed} đã thay đổi sau khi chấm.")
        checkpoint_path = Path(meta["checkpoint"])
        if sha256_file(checkpoint_path) != meta["checkpoint_sha256"]:
            raise ValueError(f"Checkpoint seed {seed} đã thay đổi sau khi đánh giá.")
        source_hashes[path] = _OBSERVED_FILES[str(output_path(path))]
        source_hashes[prediction_path] = meta["predictions_sha256"]
        source_hashes[checkpoint_path] = meta["checkpoint_sha256"]
        seed_metrics[seed] = row

    statistics = {}
    for name in METRIC_NAMES:
        values = np.asarray([seed_metrics[s][name] for s in ALL_SEEDS], dtype=np.float64)
        statistics[name] = {"mean": float(values.mean()),
                            "std": float(values.std(ddof=1)),
                            "values": {str(s): float(v) for s, v in zip(ALL_SEEDS, values)}}
    summary = {
        "_meta": owned_meta(None, "summary"),
        "metrics_sha256": {str(s): source_hashes[
            seed_dir(s) / f"metrics_seed{s}.json"] for s in ALL_SEEDS},
        "seeds": ALL_SEEDS, "n_seeds": 3, "std_ddof": 1,
        "std_definition": "sample standard deviation across three seeds, not confidence interval",
        "metric_scale": "raw pycocoevalcap scores; multiply ALL values by 100 only if desired",
        "test_images": 5000, "decode": DECODE_CONFIG,
        "protocol_hash": PROTOCOL_HASH, "protocol": PROTOCOL,
        "training_note": "All existing Level4 seeds42/44/46 reused; conditional stage fine-tuning from same Lv3seed42.",
        "old_decoder_CIDEr_raw": {str(s): HISTORICAL_METRICS[s]["CIDEr"] for s in ALL_SEEDS},
        "comparison_note": "All seeds rescored with Java METEOR; old NLTK METEOR is not interchangeable.",
        "metrics": statistics,
    }
    for path, expected_hash in source_hashes.items():
        if sha256_file(path) != expected_hash:
            raise RuntimeError(f"Input tổng hợp vừa đổi; giữ nguyên summary: {path.name}")
    summary_path = RUN_DIR / "level4_3seeds_beam3_summary.json"
    existing = read_owned_json(summary_path, None, "summary")
    if existing is not None:
        if existing != summary:
            raise ValueError("Summary có sẵn khác nội dung; giữ nguyên, không ghi đè.")
        print("Summary đã có và khớp; không sửa file.")
    else:
        save_json(summary, summary_path)

    labels = {"Bleu_1": "BLEU-1", "Bleu_2": "BLEU-2", "Bleu_3": "BLEU-3",
              "Bleu_4": "BLEU-4", "METEOR": "METEOR", "ROUGE_L": "ROUGE-L",
              "CIDEr": "CIDEr", "SPICE": "SPICE"}
    markdown = [
        "| Chỉ số (thang gốc) | Seed 42 | Seed 44 | Seed 46 | Mean ± Std |",
        "|:--|--:|--:|--:|--:|",
    ]
    for name in METRIC_NAMES:
        st = statistics[name]
        cells = [labels[name]] + [f"{seed_metrics[s][name]:.4f}" for s in ALL_SEEDS]
        cells.append(f"{st['mean']:.4f} ± {st['std']:.4f}")
        markdown.append("| " + " | ".join(cells) + " |")
    display(Markdown("\n".join(markdown)))
    display(Markdown(
        "**Quy ước:** điểm gốc của pycocoevalcap, Std mẫu với ddof=1, n=3. "
        "Đây là biến thiên giữa các seed, không phải khoảng tin cậy. "
        "Cả ba seed dùng checkpoint Level4 đã train từ cùng init Lv3 seed42. "
        "METEOR cả ba seed dùng cùng Java scorer, khác cách tính NLTK cũ."
    ))

    latex = [
        r"\begin{table}[htbp]", r"\centering",
        r"\caption{Level 4 Gated Fusion on COCO 2014 Karpathy Test (5,000 images), "
        r"beam size 3, $\alpha=0.6$. Raw scores; mean $\pm$ sample standard deviation ($n=3$).}",
        r"\label{tab:level4-multiseed-beam3}",
        r"\begin{tabular}{lrrrr}", r"\hline",
        r"Metric & Seed 42 & Seed 44 & Seed 46 & Mean $\pm$ Std \\",
        r"\hline",
    ]
    for name in METRIC_NAMES:
        st = statistics[name]
        cells = [labels[name]] + [f"{seed_metrics[s][name]:.4f}" for s in ALL_SEEDS]
        cells.append("$" + f"{st['mean']:.4f} \\pm {st['std']:.4f}" + "$")
        latex.append(" & ".join(cells) + r" \\")
    latex += [
        r"\hline", r"\end{tabular}", r"\par\smallskip",
        r"{\footnotesize Existing Level4 checkpoints; conditional stage seeds. All use Java METEOR.}",
        r"\end{table}",
    ]
    print("\nLaTeX để dán vào báo cáo:\n")
    print("\n".join(latex))
    print("\nĐã lưu:", summary_path)

missing_seeds = []
for seed in (42, 44, 46):
    folder = seed_dir(seed)
    metric_file = folder / f"metrics_seed{seed}.json"
    pred_file = folder / f"predictions_seed{seed}.json"
    if metric_file.is_file() and pred_file.is_file():
        print(f"Seed {seed}: có kết quả — sẽ xác minh hash/trạng thái trước khi gộp.")
    else:
        missing_seeds.append(seed)
        print(f"Seed {seed}: chưa có đầy đủ kết quả cuối. Thư mục:", folder)

if missing_seeds:
    print("\nChưa tổng hợp; đây là trạng thái bình thường khi chạy riêng từng seed.")
    print("Đổi SEED_TO_RUN ở Cell 2 để chạy các seed còn thiếu:", missing_seeds)
    print("Dữ liệu/điểm cũ không được sửa. Chạy lại Cell 6 sau khi đủ 42, 44, 46.")
else:
    with writing_scope("summary"):
        summarize_completed_seeds()


Seed 42: có kết quả — sẽ xác minh hash/trạng thái trước khi gộp.
Seed 44: chưa có đầy đủ kết quả cuối. Thư mục: /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level4/multiseed_beam3_v1/run_0ba507b683ac43ef/seed_44
Seed 46: chưa có đầy đủ kết quả cuối. Thư mục: /content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data/baseline_level4/multiseed_beam3_v1/run_0ba507b683ac43ef/seed_46

Chưa tổng hợp; đây là trạng thái bình thường khi chạy riêng từng seed.
Đổi SEED_TO_RUN ở Cell 2 để chạy các seed còn thiếu: [44, 46]
Dữ liệu/điểm cũ không được sửa. Chạy lại Cell 6 sau khi đủ 42, 44, 46.
